# 07 · Security and poisoning lab
The attacks run on **temporary copies**. The hash check is cryptographic and catches every change; the statistical detectors are advisory and are honest about what they miss. The contrast is why cryptographic provenance exists.

In [1]:
import json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from synera_core.config import find_repo_root, get_settings
warnings.filterwarnings("ignore")
ROOT = find_repo_root()
C = dict(bg="#061424", panel="#0b2037", cyan="#16B5FF", green="#6FD65B", amber="#FFB547", red="#FF4D5E", teal="#19E3B1", text="#EAF2FB", muted="#8FA6BF")
plt.rcParams.update({"figure.facecolor": C["bg"], "axes.facecolor": C["bg"], "axes.edgecolor": C["muted"], "axes.labelcolor": C["text"],
    "text.color": C["text"], "xtick.color": C["muted"], "ytick.color": C["muted"], "grid.color": "#1c3a5c", "axes.grid": True, "grid.linewidth": 0.5,
    "axes.prop_cycle": plt.cycler(color=[C["cyan"], C["green"], C["amber"], C["red"], C["teal"]]), "figure.dpi": 110, "font.size": 9})
SEED = get_settings().seeds.global_seed
RES = ROOT / "artifacts" / "results"

In [2]:
from synera_core.security.tamper_sim import simulate_attack, MODES
rows = []
for m in [x for x in MODES if x not in ("swap_model", "ledger_tamper")]:
    r = simulate_attack("benchmark_track_b", m, magnitude=0.05, seed=1)
    rows.append({"attack": m, "hash changed": not r["hash_match"], "statistical detectors fired": ", ".join(r["statistical_detections"]) or "none", "hash only": r["hash_only"], "verdict": r["verdict"]})
pd.DataFrame(rows)

,attack,hash changed,statistical detectors fired,hash only,verdict
0,bit_flip,True,none,True,RESULT BLOCKED FROM VERIFIED STATUS
1,subtle_edit,True,none,True,RESULT BLOCKED FROM VERIFIED STATUS
2,row_edit,True,physical_range,False,RESULT BLOCKED FROM VERIFIED STATUS
3,inject_outliers,True,physical_range,False,RESULT BLOCKED FROM VERIFIED STATUS
4,drift_units,True,physical_range,False,RESULT BLOCKED FROM VERIFIED STATUS
5,poison_labels,True,label_flip,False,RESULT BLOCKED FROM VERIFIED STATUS


In [3]:
r = simulate_attack("benchmark_track_b", "swap_model", seed=0)
print("model checks after weight perturbation:"); [print(f"  {'PASS' if c['ok'] else 'FAIL'}  {c['name']}: {c['detail']}") for c in r["model"]["after"]["checks"]]
r = simulate_attack("benchmark_track_b", "ledger_tamper")
print("\nledger copy before:", r["ledger"]["before"]["valid"], "| after one payload edit:", r["ledger"]["after"]["valid"], [i["code"] for i in r["ledger"]["after"]["issues"]][:3])

model checks after weight perturbation:
  FAIL  weights_hash: registered 6cb0f376bc19..., found 0b139d88e860...
  FAIL  holdout_metric: hold-out RMSE 3.22811 vs registered 3.22795 (tolerance 1e-06)
  FAIL  probe_predictions: predictions on the fixed probe set changed



ledger copy before: True | after one payload edit: False ['PAYLOAD_TAMPERED']


## Measured detector reliability (benchmark.json)

In [4]:
S = json.loads((RES / "benchmark.json").read_text())["security"]
print("clean false-alarm rate per detector:", {k: round(v, 3) for k, v in S["clean_false_alarm_rate"].items()})
tbl = pd.DataFrame({f"{m} {int(float(f)*100)}%": {k: v for k, v in c.items()} for m, d in S["attacks"].items() for f, c in d.items()}).T.round(2); display(tbl)
print("attacks caught by the hash but by no statistical detector:"); display(pd.DataFrame(S["hash_only_attacks"]))

clean false-alarm rate per detector: {'physical_range': 0.0, 'duplicates': 0.0, 'isolation_forest': 0.0, 'mcd': 0.0, 'autoencoder': 0.0, 'distribution_drift': 0.0, 'label_flip': 0.0, 'any_detector_warns': 0.0}


,hash,physical_range,duplicates,isolation_forest,mcd,autoencoder,distribution_drift,label_flip,any_statistical
subtle_edit 1%,1.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00
subtle_edit 5%,1.0,0.03,0.0,0.0,0.0,0.0,0.0,0.0,0.03
subtle_edit 10%,1.0,0.03,0.0,0.0,0.0,0.0,0.0,0.0,0.03
subtle_edit 20%,1.0,0.07,0.0,0.0,0.0,0.0,0.0,0.0,0.07
row_edit 1%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00
row_edit 5%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00
row_edit 10%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00
row_edit 20%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00
inject_outliers 1%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00
inject_outliers 5%,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00


attacks caught by the hash but by no statistical detector:


,mode,contamination,best_statistical_detection
0,subtle_edit,0.01,0.000000
1,subtle_edit,0.05,0.033333
2,subtle_edit,0.10,0.033333
3,subtle_edit,0.20,0.066667
4,poison_labels,0.01,0.000000
5,poison_labels,0.05,0.000000
6,poison_labels,0.10,0.000000
7,poison_labels,0.20,0.000000
